In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/raneemrefaie/pre-interview-acceptance/logatta.csv')
df

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.isnull()

In [ ]:
df.shape

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder()
df[['BusinessTravel', 'MaritalStatus', 'OverTime']] = encoder.fit_transform(df[['BusinessTravel', 'MaritalStatus', 'OverTime']])

In [ ]:
df

In [ ]:
for i in ['OverTime', 'Gender']:
    tmp_df = pd.get_dummies(df[i])
    tmp_df = tmp_df.add_prefix(f'{i}_')
    df = pd.concat([df, tmp_df], axis=1)
    df = df.drop(columns=[i])

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
for i in ['DailyRate', 'Age']:
    col_scaled = scaler.fit_transform(df[[i]])
    df[i] = col_scaled

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x=df['accepted for the interview'], hue=df['accepted for the interview'], palette='viridis', legend=False)
plt.title('Target Distribution (Accepted vs Rejected)', fontsize=12, fontweight='bold')
plt.xlabel('Interview Acceptance Status')
plt.ylabel('Candidate Count')
plt.xticks([0, 1], ['Rejected (0)', 'Accepted (1)'])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(df['DailyRate'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Distribution of Daily Rate', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Daily Rate')

sns.histplot(df['Age'], kde=True, ax=axes[1], color='orange')
axes[1].set_title('Distribution of Candidate Age', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Age')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
# Drop unique identifier 'EmployeeNumber' before correlation analysis
corr_df = df.drop(columns=['EmployeeNumber'], errors='ignore')

sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.boxplot(x=df['accepted for the interview'], y=df['DailyRate'], data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Daily Rate vs Interview Acceptance', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['Rejected', 'Accepted'])

sns.boxplot(x=df['accepted for the interview'], y=df['Age'], data=df, ax=axes[1], palette='Pastel1')
axes[1].set_title('Age vs Interview Acceptance', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['Rejected', 'Accepted'])

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

if 'EmployeeNumber' in df.columns:
    df = df.drop(columns=['EmployeeNumber'])

target_col = 'accepted for the interview'

# Define categorical features for encoding
categorical_cols = ['BusinessTravel', 'Education', 'MaritalStatus', 'OverTime', 'Gender']
numerical_cols = ['Age', 'DailyRate']

# Features & Target split
X = df.drop(columns=[target_col])
y = df[target_col]

# Split the data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features using StandardScaler
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = {}
predictions = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Test & Predict
    y_pred = model.predict(X_test_scaled)
    acc_score = model.score(X_test_scaled, y_test)
    
    # Save Results
    results[name] = acc_score
    predictions[name] = y_pred
    
    print(f'--- {name} Results ---')
    print(f'Accuracy Score: {acc_score:.4f}\n')
    print('Classification Report:')
    print(classification_report(y_test, y_pred))
    print('=' * 50)

In [ ]:
# Visualization: Model Performance Comparison
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

# Chart 1: Accuracy Comparison
axes[0].bar(results.keys(), results.values(), color=['#2b5c8f', '#3690c0', '#67a9cf'])
axes[0].set_title('Model Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy Score')
axes[0].set_ylim(0, 1.0)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height() - 0.08),
                     ha='center', va='center', color='white', fontweight='bold')

# Charts 2-4: Confusion Matrices
for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx + 1], cbar=False)
    axes[idx + 1].set_title(f'{name}\nConfusion Matrix', fontsize=12, fontweight='bold')
    axes[idx + 1].set_xlabel('Predicted Label')
    axes[idx + 1].set_ylabel('Actual Label')

plt.tight_layout()
plt.show()